# 🚆 สถิติผู้โดยสารระบบขนส่งสาธารณะในประเทศไทย
**วิเคราะห์ปริมาณการเดินทางของประชาชนด้วยระบบขนส่งสาธารณะ**

**ชุดข้อมูล:** สถิติผู้โดยสารรายวันของระบบขนส่งสาธารณะทั่วประเทศไทย (ปี 2568–2569)  
**ช่วงเวลา:** ประมาณ 14 เดือน  
**แหล่งข้อมูล:** กระทรวงคมนาคม

---

## วัตถุประสงค์การวิเคราะห์
1. **สัดส่วนการใช้ระบบขนส่ง (Modal Share)** — ระบบใดมีผู้โดยสารมากที่สุด?
2. **การเปรียบเทียบรถไฟฟ้าในเมือง (Urban Rail Comparison)** — พฤติกรรมผู้โดยสารแต่ละสายแตกต่างกันอย่างไร?
3. **การตรวจจับเหตุการณ์พิเศษ (Event Detection)** — วันหยุดและเทศกาลปรากฏในข้อมูลหรือไม่?
4. **การพยากรณ์ (Forecasting)** — คาดการณ์ความต้องการผู้โดยสาร 30 วันข้างหน้าด้วย Facebook Prophet

In [3]:
# ติดตั้งแพ็กเกจที่จำเป็น (รันครั้งเดียวใน Colab)
!pip install prophet plotly scikit-learn -q

zsh:1: command not found: pip


In [4]:
# นำเข้าไลบรารีทั้งหมดที่ใช้ในการวิเคราะห์
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import warnings
warnings.filterwarnings('ignore')

print('✅ นำเข้าไลบรารีสำเร็จ')

/Users/sarat/Code/superai-se6/superai-se6-mini-hackathon-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ นำเข้าไลบรารีสำเร็จ


---
## เฟส 1 — โหลดข้อมูล (Data Loading)

### เป้าหมาย
โหลดชุดข้อมูลและตรวจสอบโครงสร้างเบื้องต้น

**ชุดข้อมูลที่ใช้:**
- `passengers68.csv` — ข้อมูลผู้โดยสารปี 2568
- `passengers69.csv` — ข้อมูลผู้โดยสารปี 2569

In [ ]:
# โหลดชุดข้อมูลทั้งสองไฟล์
df68 = pd.read_csv('https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers68.csv')
df69 = pd.read_csv('https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers69.csv')

# รวมข้อมูลเป็น DataFrame เดียว
df = pd.concat([df68, df69], ignore_index=True)

print(f'df68 จำนวนแถว: {df68.shape[0]:,}  คอลัมน์: {df68.shape[1]}')
print(f'df69 จำนวนแถว: {df69.shape[0]:,}  คอลัมน์: {df69.shape[1]}')
print(f'รวมทั้งหมด:   {df.shape[0]:,}  คอลัมน์: {df.shape[1]}')
df.head()

HTTPError: HTTP Error 429: Too Many Requests

In [ ]:
# ตรวจสอบโครงสร้างข้อมูล
print('=== ข้อมูลโครงสร้าง DataFrame ===')
print(f'คอลัมน์ทั้งหมด: {df.columns.tolist()}')
print()
print('ชนิดข้อมูลแต่ละคอลัมน์:')
print(df.dtypes)
print()

# ตรวจสอบค่าไม่ซ้ำของรูปแบบการเดินทาง
print('รูปแบบการเดินทางที่มีในข้อมูล (รูปแบบการเดินทาง):')
print(df['รูปแบบการเดินทาง'].unique())
print()

# ตรวจสอบยานพาหนะทั้งหมด
print('ยานพาหนะ/ท่าที่มีในข้อมูล (ยานพาหนะ/ท่า):')
print(df['ยานพาหนะ/ท่า'].unique())